# let's map the 2.21 sinr model for 43188 (american pika)


In [3]:
import os
from pathlib import Path
import sys

import h3
import h3pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk
from shapely.geometry import shape
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score
import tensorflow as tf
from tqdm.auto import tqdm

2025-05-20 07:13:35.389630: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-20 07:13:35.389945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-20 07:13:35.394275: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-20 07:13:35.408696: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-20 07:13:37.053164: W tensorflow/compiler/tf2

In [4]:
sys.path.append('..')
from lib.models.tf_gp_model import TFGeoPriorModel
from lib.models.encoders import CoordEncoder

In [5]:
GEOMODEL_PATH = "/home/alex/inatGeoModelTraining/saved_models/saved_model_2_21_coords_elev.h5"
ELEVATION_RASTER_PATH = "/home/alex/sinr/data/env/elev_scaled.npy"
INPUT_ENCODING = "sinusoidal"
TAX_2_21_FILE = "/data-ssd/alex/datasets/vision-export-20250216070017-aka-2.21/taxonomy.csv"
THRESHOLDS = "/home/alex/inatVisionApi/geo_model_eval_output/v2_21_sinr/thresholds.csv"
TARGET_TAXON_ID = 43188

In [6]:
model = TFGeoPriorModel(GEOMODEL_PATH)
raster = np.load(ELEVATION_RASTER_PATH).astype(np.float32)
encoder = CoordEncoder(INPUT_ENCODING, raster=raster)
tax_2_21 = pd.read_csv(TAX_2_21_FILE)
inat_standard_thresholds = pd.read_csv(THRESHOLDS)

2025-05-20 07:13:38.392859: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-05-20 07:13:38.392916: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:129] retrieving CUDA diagnostic information for host: euclid
2025-05-20 07:13:38.392924: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:136] hostname: euclid
2025-05-20 07:13:38.393254: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:159] libcuda reported version is: 570.133.7
2025-05-20 07:13:38.393285: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] kernel reported version is: 570.133.7
2025-05-20 07:13:38.393292: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:241] kernel version seems to match DSO: 570.133.7


In [7]:
# get all h3 cells 
res0_cells = h3.get_res0_indexes()
all_cells = set()
for res0 in tqdm(res0_cells):
    children = h3.h3_to_children(res0, 4)
    all_cells.update(children)
    
cells_df = pd.DataFrame({"h3_04": list(all_cells)})
cells_df.set_index("h3_04", inplace=True)

  0%|          | 0/122 [00:00<?, ?it/s]

In [8]:
dfh3 = cells_df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [9]:
target_class_id = int(tax_2_21[tax_2_21.taxon_id==TARGET_TAXON_ID].iloc[0].leaf_class_id)

In [10]:
locs = dfh3[["lng", "lat"]].values
encoded_locs = encoder.encode(locs)
geo_model_features = model.get_loc_emb(encoded_locs)
preds = model.eval_one_class_from_feats(
    geo_model_features,
    class_of_interest=target_class_id
)
dfh3["preds"] = preds[0]
dfh3.reset_index(inplace=True)

In [11]:
locs = dfh3[["lng", "lat"]].values
encoded_locs = encoder.encode(locs)
geo_model_features = model.get_loc_emb(encoded_locs)
preds = model.eval_one_class_from_feats(
    geo_model_features,
    class_of_interest=target_class_id
)
dfh3["preds"] = preds[0]
dfh3.reset_index(inplace=True)

In [12]:
threshold = inat_standard_thresholds[inat_standard_thresholds.taxon_id==TARGET_TAXON_ID].iloc[0].thres
threshold

0.07488097

In [13]:
layer = pdk.Layer(
    "H3HexagonLayer",
    dfh3[dfh3.preds>threshold],
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[0, 255, 255]",
    get_line_color="[0, 255, 255]",
    line_width_min_pixels=2,
)
view_state = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-90.4194155, 
    zoom=2, 
    bearing=0, 
    pitch=0
)
r = pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state, 
)
r.show()

In [14]:
# let's load the lpt r thresholds

In [20]:
lpt_r_thresholds = pd.read_csv(
    "/home/alex/inatGeoModelTraining/thresholds/thresh_inat_2_21_lpt_r_10.csv"
)
lpt_r_thresholds

,taxon_id,threshold_type,threshold_value
0,152875,lpt_r_at_10_pct,0.008149
1,48272,lpt_r_at_10_pct,0.017466
2,81614,lpt_r_at_10_pct,0.003223
3,712964,lpt_r_at_10_pct,0.002584
4,472689,lpt_r_at_10_pct,0.034744
...,...,...,...
101369,783080,lpt_r_at_10_pct,0.004393
101370,1145193,lpt_r_at_10_pct,0.002462
101371,1538530,lpt_r_at_10_pct,0.062517
101372,1538531,lpt_r_at_10_pct,0.004039


In [22]:
lpt_r_thresh = lpt_r_thresholds[lpt_r_thresholds.taxon_id==TARGET_TAXON_ID].iloc[0].threshold_value
lpt_r_thresh

0.0538335874676704

In [29]:
layer = pdk.Layer(
    "H3HexagonLayer",
    dfh3[dfh3.preds>0.15],
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="[0, 255, 255]",
    get_line_color="[0, 255, 255]",
    line_width_min_pixels=2,
)
view_state = pdk.ViewState(
    latitude=37.7749295, 
    longitude=-90.4194155, 
    zoom=2, 
    bearing=0, 
    pitch=0
)
r = pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state, 
)
r.show()